In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
data_dir = '../data'  # /gpfs01/euler/data/Resources/Classifier_v2
chirp_feats_file = os.path.join(data_dir, 'chirp_feats.npz')
bar_feats_file = os.path.join(data_dir, 'bar_feats.npz')
baden_data_file = os.path.join(data_dir, 'RGCData_postprocessed.mat')

In [ ]:
from gcl_classifier.baden16 import load_baden_data

data_16 = load_baden_data(baden_data_file)
data_16.keys()

In [ ]:
data_16["chirp_traces"].shape

In [ ]:
chirp_feats = np.load(chirp_feats_file)
bar_feats = np.load(bar_feats_file)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3), width_ratios=(2, 1))

ax = axs[0]
ax.imshow(chirp_feats.T, aspect='auto', cmap='viridis', interpolation='none')
ax.set_yticks(np.arange(0, chirp_feats.shape[1]))
ax.set_ylabel('Chirp features')
ax.set_xlabel('Time [frames]')

for t in [2, 5, 8, 10, 18, 20, 28, 30]:
    ax.axvline(7.8125 * t, c='k', zorder=100, ls='--')

ax = axs[1]
ax.imshow(bar_feats.T, aspect='auto', cmap='viridis', interpolation='none')
ax.set_yticks(np.arange(0, bar_feats.shape[1]))
ax.set_ylabel('Bar features')
ax.set_xlabel('Time [frames]')

for t in [1, 2, 3]:
    ax.axvline(7.8125 * t, c='k', zorder=100, ls='--')

plt.show()

In [ ]:
from gcl_classifier.classifier import extract_features

X_16, feature_names = extract_features(
    preproc_chirps=data_16['chirp_traces'],
    preproc_bars=data_16['bar_traces'],
    bar_ds_pvalues=data_16['bar_dp'],
    roi_size_um2s=data_16['roi_size_um2'],
    chirp_features=chirp_feats,
    bar_features=bar_feats,
)

# Train new classifier

In [ ]:
# Pick level
y_16 = np.array([data_16['cluster_labels']]).flatten()
y_16.shape

In [ ]:
unique_clusters = np.unique(y_16)

In [ ]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(X_16))

train_idx, test_idx = train_test_split(indices, test_size=0.1, stratify=y_16, random_state=5)

X_train = X_16[train_idx]
X_test = X_16[test_idx]
y_train = y_16[train_idx]
y_test = y_16[test_idx]

## HP tuning

This code implements a semi-automatic HP search.
HP search space was optimized through multiple steps.

In [ ]:
def plot_search(search):
    results_df = pd.DataFrame(search.cv_results_)
    params_df = results_df['params'].apply(pd.Series)
    plot_df = pd.concat([params_df, results_df['mean_test_score']], axis=1)

    # Set up the plotting grid: one subplot per parameter
    num_params = params_df.shape[1]
    fig, axes = plt.subplots(nrows=1, ncols=num_params, figsize=(12, 3), sharey=True)

    if num_params == 1:
        axes = [axes]  # Ensure axes iterable even if only 1 param

    for ax, param in zip(axes, params_df.columns):
        sns.boxplot(x=param, y='mean_test_score', data=plot_df, ax=ax)

        ax.set_title(f'{param}')
        ax.set_xlabel(param)
        ax.set_ylabel('Mean CV Score')
        ax.grid(axis='y')

    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

## Random fit

In [ ]:
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 4, 8],
    'max_features': [10, 6, 4],
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
)

rand_search = RandomizedSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=10,
    verbose=2,
    n_iter=20,
    random_state=0,
)

rand_search.fit(X_train, y_train)

print("Best Parameters:", rand_search.best_params_)
print("Best CV Score:", rand_search.best_score_)

plot_search(rand_search)

## Grid search on narrower grid

In [ ]:
param_grid = {
    'n_estimators': [300],
    'max_depth': [20, 25, 30],
    'min_samples_split': [4, 5, 6],
    'min_samples_leaf': [2],
    'max_features': [3, 4, 5],
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
)

grid_search1 = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=10,
    verbose=1,
)

grid_search1.fit(X_train, y_train)

print("Best Parameters:", grid_search1.best_params_)
print("Best CV Score:", grid_search1.best_score_)

plot_search(grid_search1)

## More tweaking

In [ ]:
param_grid = {
    'ccp_alpha': [0, 0.00001, 0.0001, 0.001],
    'n_estimators': [300, 600, 1000],
    'max_depth': [20],
    'min_samples_split': [5],
    'min_samples_leaf': [2],
    'max_features': [4],
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)
grid_search2 = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=10,
    verbose=1,
)

grid_search2.fit(X_train, y_train)

print("Best Parameters:", grid_search2.best_params_)
print("Best CV Score:", grid_search2.best_score_)

plot_search(grid_search2)

# Define best hyper-parameters

In [ ]:
# Let's pick the highest ccp_alpha that does not impact performance
best_rf_params = {'ccp_alpha': 0.0001, 'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 2,
                  'min_samples_split': 5, 'n_estimators': 600}

# Check robustness to different seeds

In [ ]:
# Sanity check: Make sure it's not strongly seed dependent
param_grid = {
    'random_state': [42, 123182, 123, 987, 564],
}

rf = RandomForestClassifier(**best_rf_params)

grid_search3 = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=10,
    verbose=2,
)

grid_search3.fit(X_train, y_train)

print("Best Parameters:", grid_search3.best_params_)
print("Best CV Score:", grid_search3.best_score_)

plot_search(grid_search3)

## Fit with best params, calibrate and evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score


def plot_confusion_matrix(y, y_pred):
    cm = confusion_matrix(y, y_pred)
    cm = cm / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    sns.heatmap(cm, ax=ax, cmap='Blues', cbar=True)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.show()
    return fig, ax

In [ ]:
best_rf_params

In [ ]:
uncalibrated_clf = RandomForestClassifier(
    **best_rf_params,
    random_state=42,
)

uncalibrated_clf.fit(X_train, y_train)

In [ ]:
y_train_pred = uncalibrated_clf.predict(X_train)
y_test_pred = uncalibrated_clf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

fig, ax = plot_confusion_matrix(y_train, y_train_pred)
fig.suptitle('Training set')
plot_confusion_matrix(y_test, y_test_pred)
fig.suptitle('Test set')
plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

rf = RandomForestClassifier(
    **best_rf_params,
    random_state=765,
)

calibrated_clf = CalibratedClassifierCV(
    estimator=rf,
    method="sigmoid",
    cv=5,
)

calibrated_clf.fit(X_train, y_train)

In [ ]:
y_train_pred = calibrated_clf.predict(X_train)
y_test_pred = calibrated_clf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

fig, ax = plot_confusion_matrix(y_train, y_train_pred)
fig.suptitle('Training set')
plot_confusion_matrix(y_test, y_test_pred)
fig.suptitle('Test set')
plt.show()

# Fit final classifier on all data

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

rf = RandomForestClassifier(
    **best_rf_params,
    random_state=765,
)

calibrated_clf_full = CalibratedClassifierCV(
    estimator=rf,
    method="sigmoid",
    cv=5,
)

calibrated_clf_full.fit(X_16, y_16)

In [ ]:
y_16_pred = calibrated_clf_full.predict(X_16)

train_16_acc = accuracy_score(y_16, y_16_pred)

print(f"Complete dataset accuracy: {train_16_acc:.3f}")

fig, ax = plot_confusion_matrix(y_16, y_16_pred)
fig.suptitle('Complete dataset')
plt.show()

# Visualize results

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


def create_consecutive_colormap():
    """
    Create a consecutive colormap for values 1-75 with smooth transitions:
    1-13: red - OFF
    14-20: orange - ON-OFF
    21-29: green - Fast ON
    30-41: blue - Slow ON (including 2 uncertain)
    42-49: violet - SbC
    50-75: gray - ACs
    """

    # Define anchor colors at transition points
    # Positions calculated as (value - 1) / 74 to center on class boundaries
    colors_and_positions = [
        (0.0, '#8B0000'),  # 1: Dark red (start)
        ((13.5 - 1) / 74, '#DC143C'),  # 13.5: Red center
        ((20.5 - 1) / 74, '#FF8C00'),  # 20.5: Orange center
        ((29.5 - 1) / 74, '#228B22'),  # 29.5: Green center
        ((40.5 - 1) / 74, '#4169E1'),  # 40.5: Blue center
        ((45.5 - 1) / 74, '#8A2BE2'),  # 45.5: Violet center
        ((62.5 - 1) / 74, '#A9A9A9'),  # 62.5: Gray center
        (1.0, '#D3D3D3')  # 75: Light gray (end)
    ]

    # Extract colors and positions
    positions = [pos for pos, color in colors_and_positions]
    colors = [color for pos, color in colors_and_positions]

    # Create the colormap
    custom_cmap = LinearSegmentedColormap.from_list('custom',
                                                    list(zip(positions, colors)),
                                                    N=256)

    return custom_cmap


def get_color_for_value(value, cmap, vmin=1, vmax=75):
    """Get the specific color for a given value"""
    if not (vmin <= value <= vmax):
        raise ValueError(f"Value must be between {vmin} and {vmax}")

    normalized_value = (value - 1) / (vmax - vmin)
    color = cmap(normalized_value)

    return color


smooth_cmap = create_consecutive_colormap()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_16_scaled = scaler.fit_transform(X_16)

tsne = TSNE(n_components=2, perplexity=30)
X_16_tsne = tsne.fit_transform(X_16_scaled)

In [ ]:
df_tsne = pd.DataFrame({'x': X_16_tsne[:, 0], 'y': X_16_tsne[:, 1], 'cluster': y_16})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df_tsne, x='x', y='y', hue='cluster', palette=smooth_cmap, ax=ax, alpha=0.8, legend='full')
ax.legend(ncols=4, bbox_to_anchor=(1, 1), loc='upper left')
ax.set_title('t-SNE')
plt.show()

In [ ]:
n_rows = int(np.ceil((unique_clusters.size) ** 0.5))

fig, axs = plt.subplots(n_rows, n_rows, figsize=(15, 15), sharey='all', sharex='all')
axs = axs.flatten()

_df_scatter = df_tsne.sample(frac=0.2)

for i, ax in enumerate(axs):
    sns.scatterplot(ax=ax, data=_df_scatter, x='x', y='y', alpha=0.8, legend=False, color='k', s=1)
    if i < unique_clusters.size:
        cluster = unique_clusters[i]
        sns.kdeplot(ax=ax, data=df_tsne[df_tsne.cluster == cluster], x='x', y='y',
                    color=get_color_for_value(cluster, smooth_cmap), alpha=0.8, levels=[0.25, 0.5, 0.75])
        ax.text(df_tsne.x.min(), df_tsne.y.max(), str(cluster), ha='left', va='top')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df_tsne, x='x', y='y', color='k', s=2, ax=ax)

unique_clusters = np.unique(df_tsne.cluster)
for cluster in unique_clusters:
    sns.kdeplot(ax=ax, data=df_tsne[df_tsne.cluster == cluster], x='x', y='y',
                color=get_color_for_value(cluster, smooth_cmap), alpha=0.8, levels=[0.9])

plt.show()

# Save classifier to file

## Pickle, saving all data

In [ ]:
from gcl_classifier.baden16 import baden_cluster_id_to_cluster_name
from gcl_classifier.classifier import save_classifier_and_data

save_classifier_and_data(
    classifier=calibrated_clf_full,
    chirp_feats=chirp_feats,
    bar_feats=bar_feats,
    feature_names=feature_names,
    train_x=X_16,
    train_y=y_16,
    y_names={cluster: baden_cluster_id_to_cluster_name(cluster) for cluster in unique_clusters},
    classifier_file='../exports/gcl_classifier.pkl'
)

In [ ]:
import pickle

classifier_file = '../exports/gcl_classifier.pkl'

with open(classifier_file, 'rb') as f:
    classifier_dict = pickle.load(f)

## Saving weights only

In [ ]:
import joblib

joblib.dump(
    classifier_dict['classifier'],
    '../gcl_classifier/weights/gcl_classifier.joblib',
    compress=3,
)

In [ ]:
import sklearn
import skops.io as sio

# Saving with metadata
obj = {
    'model': classifier_dict['classifier'],
    'sklearn_version': sklearn.__version__,
    'metadata': {
        'description': 'Calibrated Random Forest Classifier for 2p calcium GCL responses',
        'features': feature_names,
    }
}

sio.dump(
    obj,
    '../gcl_classifier/weights/gcl_classifier.skops',
    compresslevel=9,
)

In [ ]:
len(feature_names)

## Upload to huggingface

In [ ]:
from huggingface_hub import HfApi

# Upload to Hugging Face
api = HfApi()

In [ ]:
api.upload_file(
    path_or_fileobj='../gcl_classifier/weights/gcl_classifier.skops',
    path_in_repo="gcl_classifier.skops",  # Path in the repo
    repo_id="eulerlab/gcl_classifier",
    repo_type="model",
)

print("✓ Model uploaded successfully!")